In [19]:
import random

# Medication types (categories + forms)
medication_types = [
    # Categories
    "antidepressant",
    "painkiller",
    "antibiotic",
    "antifungal",
    "antiviral",
    "antihistamine",
    "antipsychotic",
    "mood stabilizer",
    "anti-inflammatory",
    "blood thinner",
    "bronchodilator",
    "steroid",
    "hormone",
    "antacid",
    "diuretic",

    # Dosage forms
    "tablet",
    "capsule",
    "liquid",
    "syrup",
    "injection",
    "patch",
    "cream",
    "ointment",
    "inhaler",
    "spray",
    "drops",
    "gel"
]

# Optional variations / synonyms
synonyms = {
    "painkiller": ["pain killer", "analgesic"],
    "antidepressant": ["depression medicine"],
    "antibiotic": ["infection medicine"],
    "antihistamine": ["allergy medicine"],
    "anti-inflammatory": ["anti inflammatory", "inflammation medicine"],
    "blood thinner": ["anticoagulant"],
    "bronchodilator": ["asthma medicine"],
    "steroid": ["corticosteroid"],
    "tablet": ["pill"],
    "liquid": ["oral liquid"],
    "cream": ["topical cream"],
    "ointment": ["topical ointment"],
    "drops": ["eye drops", "ear drops"]
}

# Sentence patterns
patterns = [
    "it's {article} {medication_type}",
    "it is {article} {medication_type}",
    "it's in {article} {medication_type} form",
    "{article} {medication_type}",
    "{medication_type}"
]

def get_article(word):
    return "an" if word[0].lower() in "aeiou" else "a"

examples = []

for med_type in medication_types:
    all_forms = [med_type] + synonyms.get(med_type, [])

    for form in all_forms:
        article = get_article(form)
        article_cap = article.capitalize()

        for pattern in patterns:
            case_variants = [
                form.lower(),
                form.title()
            ]

            chosen = random.choice(case_variants)

            sentence = pattern.replace("{medication_type}", f"[{chosen}](medication_type)")
            sentence = sentence.replace("{article}", article)
            sentence = sentence.replace("{article_cap}", article_cap)

            examples.append(sentence)

# Shuffle the dataset
random.shuffle(examples)

from pathlib import Path

# Get current working directory (project root if you started Jupyter there)
project_root = Path.cwd().parents[1]

# Define data directory
data_dir = project_root / "data"

# Create data directory if it doesn't exist
data_dir.mkdir(parents=True, exist_ok=True)

# File path
file_path = data_dir / "medication_type_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_medication_type\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")


print(f"Generated {len(examples)} medication type examples in {file_path}")

Generated 215 medication type examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/medication_type_nlu.yml


In [2]:
import random
import re
from pathlib import Path

# List of medications (expanded and categorized)
medications = [
    # Common pain relievers
    "Paracetamol", "Acetaminophen", "Ibuprofen", "Naproxen", "Aspirin", "Diclofenac",
    
    # Diabetes
    "Metformin", "Insulin", "Insulin glargine", "Insulin aspart", "Insulin lispro",
    "Glipizide", "Glyburide", "Sitagliptin",
    
    # Antibiotics
    "Amoxicillin", "Azithromycin", "Ciprofloxacin", "Doxycycline", "Cephalexin",
    "Clindamycin", "Metronidazole", "Penicillin", "Augmentin",
    
    # Blood pressure
    "Lisinopril", "Losartan", "Amlodipine", "Metoprolol", "Atenolol", "Furosemide",
    "Hydrochlorothiazide", "Ramipril", "Valsartan",
    
    # Cholesterol
    "Atorvastatin", "Simvastatin", "Rosuvastatin", "Pravastatin",
    
    # Acid reflux
    "Omeprazole", "Pantoprazole", "Esomeprazole", "Ranitidine", "Famotidine",
    
    # Antihistamines
    "Cetirizine", "Loratadine", "Desloratadine", "Fexofenadine", "Levocetirizine",
    
    # Thyroid
    "Levothyroxine", "Liothyronine",
    
    # Mental health
    "Sertraline", "Escitalopram", "Citalopram", "Fluoxetine", "Paroxetine",
    "Venlafaxine", "Duloxetine", "Bupropion", "Mirtazapine", "Trazodone",
    "Aripiprazole", "Quetiapine", "Olanzapine", "Risperidone", "Clozapine",
    "Alprazolam", "Diazepam", "Lorazepam", "Clonazepam",
    
    # Pain management
    "Tramadol", "Codeine", "Hydrocodone", "Oxycodone", "Morphine", "Gabapentin",
    "Pregabalin",
    
    # Seizure medications
    "Carbamazepine", "Valproic acid", "Lamotrigine", "Topiramate", "Phenytoin",
    
    # Respiratory
    "Albuterol", "Salmeterol", "Fluticasone", "Budesonide", "Montelukast",
    "Tiotropium", "Formoterol",
    
    # Topical/antifungal
    "Clotrimazole", "Fluconazole", "Terbinafine", "Miconazole", "Nystatin",
    
    # Blood thinners
    "Warfarin", "Apixaban", "Rivaroxaban", "Dabigatran", "Clopidogrel",
    
    # Vitamins & supplements
    "Vitamin D", "Vitamin D3", "Vitamin B12", "Folic acid", "Calcium carbonate",
    "Magnesium glycinate", "Coenzyme Q10", "Omega-3", "Fish oil", "Multivitamin",
    
    # Steroids
    "Prednisolone", "Prednisone", "Hydrocortisone", "Dexamethasone",
    
    # Miscellaneous
    "Ondansetron", "Loperamide", "Mesalazine", "Buprenorphine", "Methadone",
    "Acetylcysteine", "Salicylic acid"
]

# Common misspellings and variations
misspellings = {
    # Pain relievers
    "Paracetamol": ["Paracetmol", "Paracitamol", "Paracetml", "PCM", "Paraceta"],
    "Acetaminophen": ["Acetamin", "Tylenol", "APAP"],
    "Ibuprofen": ["Ibuprof", "Ibuprfen", "Ibroprofen", "Brufen", "Advil", "Motrin"],
    "Naproxen": ["Naprox", "Aleve", "Naprosyn"],
    "Aspirin": ["Asprin", "Asp", "Aspirn", "ASA", "Acetylsalicylic"],
    "Diclofenac": ["Diclof", "Voltaren", "Diclofen"],
    
    # Diabetes
    "Metformin": ["Metfor", "Metfomin", "Metformn", "Glucophage", "Met"],
    "Insulin": ["Insuline", "Lantus", "Humalog", "Novolog", "Ins"],
    "Insulin glargine": ["Lantus", "Glargine"],
    "Insulin aspart": ["Novolog", "Aspart"],
    "Insulin lispro": ["Humalog", "Lispro"],
    "Glipizide": ["Glipiz", "Glucotrol"],
    "Sitagliptin": ["Januvia", "Sita"],
    
    # Antibiotics
    "Amoxicillin": ["Amox", "Amoxcillin", "Amoxacillin", "Amoxycillin", "Amoxicil"],
    "Azithromycin": ["Azithro", "Zithromax", "Azith", "Azithrom"],
    "Ciprofloxacin": ["Cipro", "Ciproxin", "Ciproflox"],
    "Doxycycline": ["Doxy", "Doxicycline", "Dox"],
    "Cephalexin": ["Cephal", "Keflex", "Cephalex"],
    "Clindamycin": ["Clinda", "Cleocin", "Clindam"],
    "Metronidazole": ["Metro", "Flagyl", "Metronida"],
    "Penicillin": ["Penic", "Peni", "Pen"],
    "Augmentin": ["Augmentine", "Augmentn", "Augment", "Co-amoxiclav"],
    
    # Blood pressure
    "Lisinopril": ["Lisinoprill", "Lisinop", "Zestril", "Prinivil", "Lysinopril"],
    "Losartan": ["Losar", "Cozaar", "Losartn"],
    "Amlodipine": ["Amlod", "Norvasc", "Amlopin", "Amlodipn"],
    "Metoprolol": ["Metop", "Lopressor", "Metopro", "Metoprol"],
    "Atenolol": ["Ateno", "Tenormin", "Atenol"],
    "Furosemide": ["Furo", "Lasix", "Furosem"],
    "Hydrochlorothiazide": ["HCTZ", "Hydrochlor", "Hydro"],
    "Ramipril": ["Rami", "Altace", "Ramip"],
    "Valsartan": ["Val", "Diovan", "Valsar"],
    
    # Cholesterol
    "Atorvastatin": ["Ator", "Lipitor", "Atorva", "Atorvastin"],
    "Simvastatin": ["Simva", "Zocor", "Simvastin"],
    "Rosuvastatin": ["Rosuva", "Crestor", "Rosuvastin"],
    "Pravastatin": ["Prava", "Pravachol", "Pravastin"],
    
    # Acid reflux
    "Omeprazole": ["Ome", "Prilosec", "Omeprazol", "Omerpazole"],
    "Pantoprazole": ["Panto", "Protonix", "Pantoprazol"],
    "Esomeprazole": ["Eso", "Nexium", "Esomeprazol"],
    "Ranitidine": ["Rani", "Zantac", "Rantidine"],
    "Famotidine": ["Famo", "Pepcid", "Famotid"],
    
    # Antihistamines
    "Cetirizine": ["Ceti", "Zyrtec", "Cetirizin"],
    "Loratadine": ["Lora", "Claritin", "Loratadin"],
    "Desloratadine": ["Deslo", "Clarinex", "Deslor"],
    "Fexofenadine": ["Fexo", "Allegra", "Fexofen"],
    "Levocetirizine": ["Levo", "Xyzal", "Levocet"],
    
    # Thyroid
    "Levothyroxine": ["Levo", "Synthroid", "Levoxyl", "Levothyr", "Levothyrox"],
    
    # Mental health
    "Sertraline": ["Sertra", "Zoloft", "Sertralin"],
    "Escitalopram": ["Escit", "Lexapro", "Escitalo"],
    "Citalopram": ["Cita", "Celexa", "Citalo"],
    "Fluoxetine": ["Fluox", "Prozac", "Fluoxetin"],
    "Paroxetine": ["Parox", "Paxil", "Paroxetin"],
    "Venlafaxine": ["Venla", "Effexor", "Venlafaxin"],
    "Duloxetine": ["Dulo", "Cymbalta", "Duloxetin"],
    "Bupropion": ["Bup", "Wellbutrin", "Bupro"],
    "Mirtazapine": ["Mirta", "Remeron", "Mirtazapin"],
    "Trazodone": ["Trazo", "Desyrel", "Trazodon"],
    "Aripiprazole": ["Ari", "Abilify", "Aripiprazol"],
    "Quetiapine": ["Queti", "Seroquel", "Quetiapin"],
    "Olanzapine": ["Olan", "Zyprexa", "Olanzapin"],
    "Risperidone": ["Risper", "Risperdal", "Risperidon"],
    "Alprazolam": ["Alpra", "Xanax", "Alpraz"],
    "Diazepam": ["Dia", "Valium", "Diazep"],
    "Lorazepam": ["Lora", "Ativan", "Lorazep"],
    "Clonazepam": ["Clona", "Klonopin", "Clonazep"],
    
    # Pain management
    "Tramadol": ["Trama", "Ultram", "Tramad"],
    "Codeine": ["Codein", "Cod"],
    "Gabapentin": ["Gaba", "Neurontin", "Gabapent"],
    "Pregabalin": ["Prega", "Lyrica", "Pregabal"],
    
    # Respiratory
    "Albuterol": ["Albu", "Ventolin", "Proair", "Salbutamol", "Albutrol"],
    "Montelukast": ["Monte", "Singulair", "Montel"],
    "Fluticasone": ["Fluti", "Flovent", "Flonase", "Flutic"],
    "Budesonide": ["Bude", "Pulmicort", "Rhinocort", "Budeson"],
    
    # Vitamins & supplements
    "Vitamin D": ["Vit D", "Vitamin D3", "Vit D3", "Cholecalciferol", "Calciferol"],
    "Vitamin D3": ["Vit D3", "Cholecalciferol"],
    "Vitamin B12": ["Vit B12", "Cobalamin", "B12"],
    "Folic acid": ["Folate", "Folic", "Vitamin B9"],
    "Calcium carbonate": ["Calcium", "CaCO3", "Cal carb"],
    "Magnesium glycinate": ["Magnesium", "Mag glycinate", "Mg glycinate"],
    "Coenzyme Q10": ["CoQ10", "Ubiquinone", "Q10"],
    "Omega-3": ["Fish oil", "Omega 3", "EPA", "DHA"],
    
    # Common brand name variations
    "Symbicort": ["Symbicortt", "Symbocort"],
    "Augmentin": ["Augmentine", "Augmentn"],
    "Amgevita": ["Amgevitta", "Amgevta"],
}

# Common phrases and patterns for mentioning medications
patterns = [
    # Direct statements
    "I take {medication_name}",
    "I'm on {medication_name}",
    "I use {medication_name}",
    "My medication is {medication_name}",
    "It's {medication_name}",
    "{medication_name}",
    
    # Adding/recording
    "Add {medication_name}",
    "Please add {medication_name}",
    "I want to add {medication_name}",
    "Can you add {medication_name}",
    "Record {medication_name}",
    "Save {medication_name}",
    "Put {medication_name} in my list",
    
    # With dosage information
    "{medication_name} 500mg",
    "{medication_name} 10mg",
    "I take {medication_name} 500mg",
    "{medication_name} 5mg daily",
    "Add {medication_name} 250mg",
    
    # Prescription/doctor related
    "My doctor prescribed {medication_name}",
    "I was prescribed {medication_name}",
    "Prescription for {medication_name}",
    "I need {medication_name}",
    
    # Questions (that might be misunderstood as providing name)
    "What is {medication_name}?",
    "Do you know {medication_name}?",
    "Have you heard of {medication_name}?",
    
    # With modifiers
    "Just {medication_name}",
    "Only {medication_name}",
    "Started {medication_name} recently",
    "Been taking {medication_name} for a while",
    
    # With symptoms/conditions
    "I take {medication_name} for pain",
    "{medication_name} helps with my condition",
    "For my blood pressure, I take {medication_name}",
]

# Additional patterns for misspellings (shorter/abbreviated)
short_forms = [
    "{medication_name}",
    "just {medication_name}",
    "it's {medication_name}",
    "med {medication_name}",
    "drug {medication_name}",
]

def clean_medication_name(name):
    """Clean medication name for consistent formatting"""
    return name.strip().lower()

def generate_medication_variations(med_name):
    """Generate all variations of a medication name including misspellings"""
    variations = [med_name]  # Original with proper case
    
    # Add lowercase
    variations.append(med_name.lower())
    
    # Add title case
    variations.append(med_name.title())
    
    # Add uppercase for acronym-like medications
    if len(med_name) < 10 and " " not in med_name:
        variations.append(med_name.upper())
    
    # Add misspellings if they exist
    if med_name in misspellings:
        for mis in misspellings[med_name]:
            variations.append(mis)
            variations.append(mis.lower())
            variations.append(mis.title())
    
    # Remove duplicates while preserving order
    seen = set()
    unique_variations = []
    for var in variations:
        if var not in seen:
            seen.add(var)
            unique_variations.append(var)
    
    return unique_variations

# Generate examples
examples = []
used_combinations = set()  # To avoid exact duplicates

for med in medications:
    med_variations = generate_medication_variations(med)
    
    for variation in med_variations:
        # Choose pattern set
        pattern_set = patterns
        if len(variation.split()) <= 2:  # Short names can use short forms
            pattern_set = patterns + short_forms
        
        for pattern in pattern_set:
            # Create the example with entity annotation
            example = pattern.replace("{medication_name}", f"[{variation}](medication_name)")
            
            # Avoid exact duplicates
            if example not in used_combinations:
                used_combinations.add(example)
                examples.append(example)
            
            # Add some examples with trailing punctuation
            if random.random() < 0.3:  # 30% chance
                punct_variations = [
                    example + ".",
                    example + ",",
                    example + "?",
                    example + "!"
                ]
                for punct_ex in punct_variations:
                    if punct_ex not in used_combinations and random.random() < 0.5:
                        used_combinations.add(punct_ex)
                        examples.append(punct_ex)

# Shuffle thoroughly
random.shuffle(examples)

# Limit to a reasonable number (avoid overwhelming the NLU)
max_examples = 2000
if len(examples) > max_examples:
    examples = random.sample(examples, max_examples)

# Sort for better readability (optional)
examples.sort()

# Write to file
project_root = Path.cwd()
while project_root.name != "pillaxia-ai" and project_root.parent != project_root:
    project_root = project_root.parent

data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "medication_name_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\n")
    f.write("nlu:\n")
    f.write("- intent: provide_medication_name\n")
    f.write("  examples: |\n")
    
    # Write examples in batches for better organization
    for i, ex in enumerate(examples):
        f.write(f"    - {ex}\n")
    

print(f"✅ Generated {len(examples)} NLU examples for 'provide_medication_name'")
print(f"📁 Saved to: {file_path}")

# Optional: Show sample
print("\n📋 Sample examples:")
for i in range(min(10, len(examples))):
    print(f"  {examples[i]}")

✅ Generated 2000 NLU examples for 'provide_medication_name'
📁 Saved to: /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/medication_name_nlu.yml

📋 Sample examples:
  Add [Allegra](medication_name)
  Add [Amoxycillin](medication_name) 250mg.
  Add [Asp](medication_name) 250mg,
  Add [Atenol](medication_name) 250mg
  Add [Atenol](medication_name),
  Add [Ativan](medication_name) 250mg
  Add [Bup](medication_name)
  Add [Buprenorphine](medication_name) 250mg
  Add [Bupro](medication_name)
  Add [Calcium Carbonate](medication_name)


In [20]:
import random
from pathlib import Path

# Instruction phrases grouped by type
instruction_groups = {
    "timing": [
        "in the morning",
        "at night",
        "before bed",
        "after meals",
        "before meals",
        "with breakfast",
        "with dinner",
        "on an empty stomach"
    ],
    "frequency": [
        "once daily",
        "twice daily",
        "three times a day",
        "every 8 hours",
        "every 12 hours",
        "once a week",
        "as needed",
        "when required"
    ],
    "food_related": [
        "with food",
        "without food",
        "after food",
        "before food"
    ],
    "duration": [
        "for 5 days",
        "for 7 days",
        "for two weeks",
        "for a month",
        "until finished"
    ],
    "route": [
        "by mouth",
        "orally",
        "topically",
        "as an injection",
        "with an inhaler"
    ]
}

# Flatten all instruction phrases
all_instructions = []
for group in instruction_groups.values():
    all_instructions.extend(group)

# Sentence patterns
patterns = [
    "I take it {instruction}",
    "I use it {instruction}",
    "I apply it {instruction}",
    "I inject it {instruction}",
    "It should be taken {instruction}",
    "It's taken {instruction}",
    "Doctor said {instruction}",
    "{instruction}",
    "I normally take it {instruction}",
    "I'm supposed to take it {instruction}"
]

examples = []

for instruction in all_instructions:
    for pattern in patterns:
        case_variants = [
            instruction.lower(),
            instruction.title()
        ]

        chosen = random.choice(case_variants)

        sentence = pattern.replace(
            "{instruction}",
            f"[{chosen}](medication_instructions)"
        )

        examples.append(sentence)

# Shuffle dataset
random.shuffle(examples)

# -------- SAVE FILE --------

# Go up to project root (helpers → actions → project_root)
project_root = Path.cwd().parents[1]

data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "medication_instructions_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_medication_instructions\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} instruction examples in {file_path}")


Generated 300 instruction examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/medication_instructions_nlu.yml


In [7]:
import random
from pathlib import Path

# Common doses and units
dose_units = ["mg", "g", "ml", "IU", "tablet", "capsule", "drop", "puff"]

# Example numbers / quantities
dose_numbers = [
    "1", "2", "3", "4", "5", "10", "20", "50", "100", "250", "500", "750", "1000"
]

# Frequency modifiers
frequencies = [
    "once daily", "twice daily", "three times a day", "every 8 hours", "every 12 hours", "as needed"
]

# Sentence patterns
patterns = [
    "I take {dose}",
    "I take {dose} {frequency}",
    "I use {dose}",
    "I use {dose} {frequency}",
    "My dose is {dose}",
    "My dose is {dose} {frequency}",
    "Doctor prescribed {dose}",
    "Doctor prescribed {dose} {frequency}",
    "{dose}",
    "{dose} {frequency}"
]

# Generate dose combinations (always number + unit)
dose_combinations = []
for number in dose_numbers:
    for unit in dose_units:
        # With space (e.g., "5 mg")
        dose_combinations.append(f"{number} {unit}")
        # Without space (e.g., "5mg")
        dose_combinations.append(f"{number}{unit}")

# Generate examples
examples = []

for dose in dose_combinations:
    for pattern in patterns:
        # Randomly decide whether to include frequency (70% chance)
        include_frequency = random.random() < 0.7
        freq = random.choice(frequencies) if include_frequency else ""
        
        # Build the sentence with the dose marked as entity
        sentence = pattern.replace("{dose}", f"[{dose}](medication_dose)")
        sentence = sentence.replace("{frequency}", freq)
        
        # Clean up extra spaces
        sentence = " ".join(sentence.split())
        examples.append(sentence)

# Shuffle dataset
random.shuffle(examples)

# Remove duplicates while preserving order
seen = set()
unique_examples = []
for ex in examples:
    if ex not in seen:
        seen.add(ex)
        unique_examples.append(ex)

# Limit to a reasonable number (optional)
max_examples = 500
if len(unique_examples) > max_examples:
    unique_examples = unique_examples[:max_examples]

# -------- SAVE FILE --------

# Go up to project root (helpers → actions → project_root)
project_root = Path.cwd().parents[1]

data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "medication_dose_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_medication_dose\n  examples: |\n")
    for ex in unique_examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(unique_examples)} dose examples in {file_path}")
print(f"Examples include only 'number + unit' formats like '5 mg' or '5mg'")


Generated 500 dose examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/medication_dose_nlu.yml
Examples include only 'number + unit' formats like '5 mg' or '5mg'


In [2]:
import random
from pathlib import Path

# Numbers users might say
stock_numbers = [
    "0", "1", "2", "3", "5", "7", "10", "12", "15", "20", "24",
    "30", "45", "60", "90", "100", "120", "200"
]

# Medication forms
med_forms = [
    "tablet", "tablets",
    "pill", "pills",
    "capsule", "capsules",
    "dose", "doses",
    "strip", "strips",
    "bottle", "bottles"
]

# Natural variations of how users express stock
patterns = [
    "I have {number} {form} left",
    "I have {number} {form} remaining",
    "There are {number} {form} left",
    "Only {number} {form} left",
    "About {number} {form} left",
    "Roughly {number} {form} remaining",
    "I’ve got {number} {form} left",
    "Still have {number} {form}",
    "Currently have {number} {form}",
    "Stock is {number} {form}",
    "{number} {form} left",
    "{number} {form} remaining",
    "Just {number} {form}",
    "Around {number} {form}",
    "I’m down to {number} {form}",
    "Only {number} remaining",
    "There’s {number} left",
    "{number} left",
    "{number}"
]

examples = []

for number in stock_numbers:
    for form in med_forms:
        for pattern in patterns:
            sentence = pattern.format(
                number=f"[{number}](stock_level)",
                form=form
            )
            examples.append(sentence)

# Shuffle for randomness
random.shuffle(examples)

# -------- SAVE FILE --------

project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "refill_stock_level_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_stock_level\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} stock level examples in {file_path}")


Generated 4104 stock level examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/refill_stock_level_nlu.yml


In [1]:
import random
from pathlib import Path

# Base numeric values in days
day_numbers = [0, 1, 2, 3, 4, 5, 7, 10, 14, 21, 30, 45, 60, 90]

# Conversion helpers
def to_weeks(days):
    if days % 7 == 0 and days != 0:
        return days // 7
    return None

def to_months(days):
    if days % 30 == 0 and days != 0:
        return days // 30
    return None


# -------- SHORT / FRAGMENT PATTERNS --------

fragment_patterns = [
    "{value}",
    "in {value}",
    "about {value}",
    "around {value}",
    "only {value}",
    "{value} left",
    "maybe {value}",
]

# -------- COMPLETE SENTENCE PATTERNS --------

sentence_patterns = [
    "I need to refill in {value}.",
    "My refill is due in {value}.",
    "I have {value} before I need a refill.",
    "There are {value} remaining before refill.",
    "My medication will run out in {value}.",
    "I will need a refill in {value}.",
    "The supply finishes in {value}.",
    "Refill coming up in {value}.",
]


examples = []

for days in day_numbers:

    # ----- DAYS -----
    day_value = f"{days} days"
    annotated_day = f"[{day_value}](refill_day)"

    for pattern in fragment_patterns:
        examples.append(pattern.format(value=annotated_day))

    for pattern in sentence_patterns:
        examples.append(pattern.format(value=annotated_day))

    # ----- WEEKS -----
    weeks = to_weeks(days)
    if weeks:
        week_label = "week" if weeks == 1 else "weeks"
        week_value = f"{weeks} {week_label}"
        annotated_week = f"[{week_value}](refill_day)"

        # numeric week examples (1 week, 2 weeks, etc.)
        for pattern in fragment_patterns:
            examples.append(pattern.format(value=annotated_week))

        for pattern in sentence_patterns:
            examples.append(pattern.format(value=annotated_week))

        # add "a week" when weeks == 1
        if weeks == 1:
            annotated_a_week = "[a week](refill_day)"

            for pattern in fragment_patterns:
                examples.append(pattern.format(value=annotated_a_week))

            for pattern in sentence_patterns:
                examples.append(pattern.format(value=annotated_a_week))

    # ----- MONTHS -----
    months = to_months(days)
    if months:
        month_label = "month" if months == 1 else "months"
        month_value = f"{months} {month_label}"
        annotated_month = f"[{month_value}](refill_day)"

        # numeric month examples (1 month, 2 months, etc.)
        for pattern in fragment_patterns:
            examples.append(pattern.format(value=annotated_month))

        for pattern in sentence_patterns:
            examples.append(pattern.format(value=annotated_month))

        # add "a month" when months == 1
        if months == 1:
            annotated_a_month = "[a month](refill_day)"

            for pattern in fragment_patterns:
                examples.append(pattern.format(value=annotated_a_month))

            for pattern in sentence_patterns:
                examples.append(pattern.format(value=annotated_a_month))


# Shuffle dataset
random.shuffle(examples)


# -------- SAVE FILE --------

project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "refill_day_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_refill_day\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} refill day examples in {file_path}")

Generated 330 refill day examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/refill_day_nlu.yml


In [ ]:
import random
from pathlib import Path

# Base frequency types and their plural forms
frequency_types = {
    "day": "days",
    "week": "weeks",
    "month": "months",
    "year": "years"
}

# Example counts (number of periods)
counts = ["1", "2", "3", "5", "10", "30"]

# Sentence patterns
patterns = [
    "Remind me for [{frequency}](frequency)",
    "I want to be reminded for [{frequency}](frequency)",
    "Set a reminder for [{frequency}](frequency)",
    "Please keep it for [{frequency}](frequency)",
    "Make it [{frequency}](frequency)",
    "I would like to be reminded for [{frequency}](frequency)",
    "[{frequency}](frequency)"
]

examples = []

for unit, plural in frequency_types.items():
    for pattern in patterns:
        count = random.choice(counts)

        # Use singular if count is 1, else plural
        unit_text = unit if count == "1" else plural

        frequency_text = f"{count} {unit_text}"

        sentence = pattern.replace("{frequency}", frequency_text)
        examples.append(sentence)

# Shuffle dataset
random.shuffle(examples)

# -------- SAVE FILE --------
project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "reminder_frequency_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_frequency\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} frequency examples in {file_path}")


Generated 28 frequency examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/reminder_frequency_nlu.yml


In [12]:
import random
from pathlib import Path

# Values for per_day_frequency
per_day_frequencies = ["once", "twice", "thrice"]

# Sentence patterns
patterns = [
    "I want to be reminded [{freq}](per_day_frequency) per day",
    "Remind me [{freq}](per_day_frequency) daily",
    "Set a reminder [{freq}](per_day_frequency) each day",
    "Please remind me [{freq}](per_day_frequency) per day",
    "Make it [{freq}](per_day_frequency) every day",
    "I would like to be reminded [{freq}](per_day_frequency) daily",
    "[{freq}](per_day_frequency)"
]

examples = []

for freq in per_day_frequencies:
    for pattern in patterns:
        sentence = pattern.replace("{freq}", freq)
        examples.append(sentence)

# Shuffle dataset
random.shuffle(examples)

# -------- SAVE FILE --------
project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "per_day_frequency_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_per_day_frequency\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} per-day frequency examples in {file_path}")

Generated 21 per-day frequency examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/per_day_frequency_nlu.yml


In [15]:
import random
from pathlib import Path

# Example quantities
quantities = ["10 mg", "5 mg", "20 mg", "50 mg", "100 mg",
              "5 ml", "10 ml", "15 ml", "20 ml",
              "100 IU", "200 IU", "500 IU", "1 tablet"]

# Sentence patterns
patterns = [
    "I want to take [{quantity}](quantity)",
    "Set my dosage to [{quantity}](quantity)",
    "Take [{quantity}](quantity) daily",
    "Remind me to take [{quantity}](quantity)",
    "My dose is [{quantity}](quantity)",
    "Please record [{quantity}](quantity) as my dose",
    "[{quantity}](quantity)"
]

examples = []

for qty in quantities:
    for pattern in patterns:
        sentence = pattern.replace("{quantity}", qty)
        examples.append(sentence)

# Shuffle dataset
random.shuffle(examples)

# -------- SAVE FILE --------
project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "reminder_quantity_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_quantity\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} provide_quantity examples in {file_path}")


Generated 91 provide_quantity examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/reminder_quantity_nlu.yml


In [ ]:
import random
from pathlib import Path

# --------------------------
# Generate times
# --------------------------

# 12-hour AM/PM format
times_12h = [f"{h} am" for h in range(1, 12)] + [f"{h} pm" for h in range(1, 12)]
# Add some with minutes
times_12h += [f"{h}:00 am" for h in range(1, 12)] + [f"{h}:30 pm" for h in range(1, 12)]
# 24-hour format
times_24h = [f"{h:02d}:00" for h in range(0, 24)] + [f"{h:02d}:30" for h in range(0, 24)]

# Friendly phrases like "8 in the morning"
friendly_times = [
    "6 in the morning", "7 in the morning", "8 in the morning", "9 in the morning",
    "10 in the morning", "11 in the morning", "12 noon",
    "1 in the afternoon", "2 in the afternoon", "3 in the afternoon",
    "4 in the afternoon", "5 in the afternoon", "6 in the evening",
    "7 in the evening", "8 at night", "9 at night"
]

all_times = times_12h + times_24h + friendly_times

# --------------------------
# Sentence patterns
# --------------------------
patterns = [
    "Remind me at {times}",
    "Set a reminder for {times}",
    "I want to be reminded at {times}",
    "Please remind me at {times}",
    "Make it {times}",
    "Schedule my reminder for {times}",
    "Notify me at {times}",
    "Remind me in the morning at {times}",
    "Remind me tonight at {times}",
    "{times}"
]

# --------------------------
# Generate examples
# --------------------------
examples = []

for num_times in [1, 2, 3]:  # 1 to 3 times per sentence
    for _ in range(15):  # multiple samples per combination
        sampled_times = random.sample(all_times, num_times)

        # Annotate each time
        annotated_times = [f"[{t}](reminder_time)" for t in sampled_times]

        # Join multiple times with commas and 'and' for realism
        if len(annotated_times) == 1:
            times_text = annotated_times[0]
        else:
            times_text = ", ".join(annotated_times[:-1]) + " and " + annotated_times[-1]

        for pattern in patterns:
            sentence = pattern.replace("{times}", times_text)
            examples.append(sentence)

# Shuffle dataset
random.shuffle(examples)

# --------------------------
# Save to file
# --------------------------
project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "reminder_time_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_reminder_time\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} provide_reminder_time examples in {file_path}")


Generated 450 provide_reminder_time examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/reminder_time_nlu.yml


In [2]:
import random
from pathlib import Path

# Possible alert sound values
alert_types = ["alarm", "voice"]

# ---- DIRECT COMMANDS ----
command_patterns = [
    "Set it to [{sound}](alert_type)",
    "Use [{sound}](alert_type)",
    "Make it [{sound}](alert_type)",
    "I want [{sound}](alert_type)",
    "Change it to [{sound}](alert_type)",
    "Please use [{sound}](alert_type)",
]

# ---- NATURAL CONFIRMATIONS ----
confirmation_patterns = [
    "[{sound}](alert_type) is fine",
    "[{sound}](alert_type) will be fine",
    "[{sound}](alert_type) is okay",
    "[{sound}](alert_type) works",
    "yeah [{sound}](alert_type)",
    "just [{sound}](alert_type)",
    "go with [{sound}](alert_type)",
    "let's do [{sound}](alert_type)",
    "I’ll take [{sound}](alert_type)",
    "that’s fine, [{sound}](alert_type)",
]

# ---- CASUAL / FRAGMENT STYLE ----
fragment_patterns = [
    "[{sound}](alert_type)",
    "maybe [{sound}](alert_type)",
    "hmm [{sound}](alert_type)",
    "probably [{sound}](alert_type)",
    "okay [{sound}](alert_type)",
    "uh [{sound}](alert_type)",
    "[{sound}](alert_type)"
]

# ---- FULL SENTENCES ----
sentence_patterns = [
    "I prefer [{sound}](alert_type) for the reminder.",
    "Can you set the reminder to [{sound}](alert_type)?",
    "Please make the alert sound [{sound}](alert_type).",
    "I'd like the notification to use [{sound}](alert_type).",
    "For the reminder, choose [{sound}](alert_type).",
]

examples = []

for sound in alert_types:
    for pattern_group in [
        command_patterns,
        confirmation_patterns,
        fragment_patterns,
        sentence_patterns,
    ]:
        for pattern in pattern_group:
            examples.append(pattern.replace("{sound}", sound))

# Shuffle dataset
random.shuffle(examples)

# -------- SAVE FILE --------
project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "reminder_alert_type_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_alert_type\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} alert sound examples in {file_path}")


Generated 56 alert sound examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/reminder_alert_type_nlu.yml


In [3]:
import random
from pathlib import Path

# Days of the week
days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Special day expressions
special_day_expressions = [
    "every day",
    "everyday",
    "weekdays",
    "weekends",
    "daily"
]

# Casual/abbreviated forms
abbreviations = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Sentence patterns
patterns = [
    "Remind me on {days}",
    "Set a reminder for {days}",
    "I want to be reminded on {days}",
    "Please remind me on {days}",
    "Make it {days}",
    "Schedule my reminder for {days}",
    "Remind me {days}",
    "{days} reminder please",
    "I need a reminder for {days}"
]

examples = []

# Generate examples for 1 to 3 specific days
for num_days in [1, 2, 3]:
    for _ in range(10):
        sampled_days = random.sample(days_of_week, num_days)
        days_text = ", ".join(f"[{d}](reminder_day)" for d in sampled_days)
        for pattern in patterns:
            sentence = pattern.replace("{days}", days_text)
            examples.append(sentence)
        # Add abbreviated versions
        abbrev_text = ", ".join(f"[{d[:3]}](reminder_day)" for d in sampled_days)
        for pattern in patterns:
            sentence = pattern.replace("{days}", abbrev_text)
            examples.append(sentence)

# Generate examples for special day expressions
for special in special_day_expressions:
    for pattern in patterns:
        sentence = pattern.replace("{days}", f"[{special}](reminder_day)")
        examples.append(sentence)

# Add casual single-day forms like "Mondays"
for day in days_of_week:
    examples.append(f"[{day}s](reminder_day)")  # e.g., "Mondays"
    examples.append(f"[{day} morning](reminder_day)")  # e.g., "Monday morning"
    examples.append(f"[{day} evening](reminder_day)")

# Shuffle dataset
random.shuffle(examples)

# -------- SAVE FILE --------
project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "reminder_day_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\nnlu:\n- intent: provide_reminder_day\n  examples: |\n")
    for ex in examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(examples)} reminder day examples in {file_path}")


Generated 606 reminder day examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/reminder_day_nlu.yml


In [2]:
import random
from pathlib import Path

# ----------------------------
# CONFIGURATION
# ----------------------------

dose_units = ["mg", "g", "ml", "IU", "tablet", "capsule", "drop", "puff"]

dose_numbers = [
    "1", "2", "3", "4", "5", "10", "20", "50", "100",
    "250", "500", "750", "1000"
]

frequencies = [
    "once daily",
    "twice daily",
    "three times a day",
    "every 8 hours",
    "every 12 hours",
    "as needed"
]

patterns = [
    "I take {dose}",
    "I take {dose} {frequency}",
    "I use {dose}",
    "I use {dose} {frequency}",
    "My dose is {dose}",
    "My dose is {dose} {frequency}",
    "Doctor prescribed {dose}",
    "Doctor prescribed {dose} {frequency}",
    "Set my dosage to {dose}",
    "Remind me to take {dose}",
    "Please record {dose} as my dose",
    "{dose}",
    "{dose} {frequency}"
]

MAX_EXAMPLES = 600

# ----------------------------
# GENERATE DOSE COMBINATIONS
# ----------------------------

dose_combinations = []

for number in dose_numbers:
    for unit in dose_units:
        dose_combinations.append(f"{number} {unit}")
        dose_combinations.append(f"{number}{unit}")

# ----------------------------
# BUILD EXAMPLES
# ----------------------------

examples = []

for dose in dose_combinations:
    for pattern in patterns:

        include_frequency = random.random() < 0.6
        freq = random.choice(frequencies) if include_frequency else ""

        # Mark entity
        tagged_dose = f"[{dose}](medication_dosage)"

        sentence = pattern.replace("{dose}", tagged_dose)
        sentence = sentence.replace("{frequency}", freq)

        sentence = " ".join(sentence.split())
        examples.append(sentence)

# ----------------------------
# REMOVE DUPLICATES
# ----------------------------

random.shuffle(examples)

seen = set()
unique_examples = []

for ex in examples:
    if ex not in seen:
        seen.add(ex)
        unique_examples.append(ex)

if len(unique_examples) > MAX_EXAMPLES:
    unique_examples = unique_examples[:MAX_EXAMPLES]

# ----------------------------
# SAVE FILE
# ----------------------------

project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "medication_dose_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\n")
    f.write("nlu:\n")
    f.write("- intent: provide_medication_dose\n")
    f.write("  examples: |\n")
    for ex in unique_examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(unique_examples)} examples in {file_path}")
print("Intent: provide_medication_dose")
print("Entity: medication_dosage")

Generated 600 examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/medication_dose_nlu.yml
Intent: provide_medication_dose
Entity: medication_dosage


In [4]:
import random
from pathlib import Path

# ----------------------------
# CONFIGURATION
# ----------------------------

day_numbers = [1, 2, 3, 4, 5, 7, 10, 14, 21, 30, 45, 60, 90]

frequency_types = {
    "day": "days",
    "week": "weeks",
    "month": "months",
    "year": "years"
}

counts = ["1", "2", "3", "5", "10", "30"]

fragment_patterns = [
    "{value}",
    "in {value}",
    "about {value}",
    "around {value}",
    "only {value}",
    "{value} left",
    "maybe {value}",
]

sentence_patterns = [
    "I need it in {value}",
    "Remind me in {value}",
    "Set it for {value}",
    "My refill is due in {value}",
    "I will need it in {value}",
    "There are {value} remaining",
    "Keep it for {value}",
    "Make it {value}",
]

MAX_EXAMPLES = 700

# ----------------------------
# HELPERS
# ----------------------------

def to_weeks(days):
    return days // 7 if days % 7 == 0 and days != 0 else None

def to_months(days):
    return days // 30 if days % 30 == 0 and days != 0 else None

# ----------------------------
# GENERATE EXAMPLES
# ----------------------------

examples = []

# ---- Generate from day numbers (refill style) ----
for days in day_numbers:

    # Days
    day_value = f"{days} days"
    annotated = f"[{day_value}](time_period)"

    for p in fragment_patterns:
        examples.append(p.format(value=annotated))

    for p in sentence_patterns:
        examples.append(p.format(value=annotated))

    # Weeks
    weeks = to_weeks(days)
    if weeks:
        label = "week" if weeks == 1 else "weeks"
        week_value = f"{weeks} {label}"
        annotated = f"[{week_value}](time_period)"

        for p in fragment_patterns:
            examples.append(p.format(value=annotated))

        for p in sentence_patterns:
            examples.append(p.format(value=annotated))

        if weeks == 1:
            annotated = "[a week](time_period)"
            for p in fragment_patterns:
                examples.append(p.format(value=annotated))
            for p in sentence_patterns:
                examples.append(p.format(value=annotated))

    # Months
    months = to_months(days)
    if months:
        label = "month" if months == 1 else "months"
        month_value = f"{months} {label}"
        annotated = f"[{month_value}](time_period)"

        for p in fragment_patterns:
            examples.append(p.format(value=annotated))

        for p in sentence_patterns:
            examples.append(p.format(value=annotated))

        if months == 1:
            annotated = "[a month](time_period)"
            for p in fragment_patterns:
                examples.append(p.format(value=annotated))
            for p in sentence_patterns:
                examples.append(p.format(value=annotated))


# ---- Generate reminder-style frequencies ----
for unit, plural in frequency_types.items():
    for pattern in sentence_patterns:

        count = random.choice(counts)
        unit_text = unit if count == "1" else plural
        value = f"{count} {unit_text}"

        annotated = f"[{value}](time_period)"
        examples.append(pattern.format(value=annotated))


# ----------------------------
# CLEAN DATASET
# ----------------------------

random.shuffle(examples)

seen = set()
unique_examples = []

for ex in examples:
    ex = " ".join(ex.split())
    if ex not in seen:
        seen.add(ex)
        unique_examples.append(ex)

if len(unique_examples) > MAX_EXAMPLES:
    unique_examples = unique_examples[:MAX_EXAMPLES]


# ----------------------------
# SAVE FILE
# ----------------------------

project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "time_period_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\n")
    f.write("nlu:\n")
    f.write("- intent: provide_time_period\n")
    f.write("  examples: |\n")
    for ex in unique_examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(unique_examples)} examples in {file_path}")
print("Intent: provide_time_period")
print("Entity: time_period")

Generated 333 examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/time_period_nlu.yml
Intent: provide_time_period
Entity: time_period


In [5]:
import random
from pathlib import Path

# ----------------------------
# CONFIGURATION
# ----------------------------

base_colours = [
    "red", "blue", "white", "yellow", "green",
    "orange", "purple", "pink", "black", "grey", "brown"
]

modifiers = [
    "light", "dark", "bright", "pale", "deep"
]

# Short answer patterns (most important for slot-filling)
short_patterns = [
    "{colour}",
    "it's {colour}",
    "it is {colour}",
    "{colour} one",
    "the {colour} one",
    "probably {colour}",
    "maybe {colour}",
]

# Full sentence patterns
sentence_patterns = [
    "The medication is {colour}",
    "It's a {colour} pill",
    "It's {colour} in color",
    "I would say {colour}",
    "Let's go with {colour}",
    "Make it {colour}",
    "Use {colour}",
]

MAX_EXAMPLES = 400

# ----------------------------
# GENERATE COLOUR VARIATIONS
# ----------------------------

colour_variations = []

for colour in base_colours:
    # Base colour
    colour_variations.append(colour)

    # Modified versions
    for mod in modifiers:
        colour_variations.append(f"{mod} {colour}")

# ----------------------------
# BUILD EXAMPLES
# ----------------------------

examples = []

for colour in colour_variations:
    tagged = f"[{colour}](medication_colour)"

    for pattern in short_patterns:
        examples.append(pattern.format(colour=tagged))

    for pattern in sentence_patterns:
        examples.append(pattern.format(colour=tagged))

# ----------------------------
# CLEAN DATASET
# ----------------------------

random.shuffle(examples)

seen = set()
unique_examples = []

for ex in examples:
    ex = " ".join(ex.split())
    if ex not in seen:
        seen.add(ex)
        unique_examples.append(ex)

if len(unique_examples) > MAX_EXAMPLES:
    unique_examples = unique_examples[:MAX_EXAMPLES]

# ----------------------------
# SAVE FILE
# ----------------------------

project_root = Path.cwd().parents[1]
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "medication_colour_nlu.yml"

with open(file_path, "w", encoding="utf-8") as f:
    f.write("version: \"3.1\"\n\n")
    f.write("nlu:\n")
    f.write("- intent: provide_medication_colour\n")
    f.write("  examples: |\n")
    for ex in unique_examples:
        f.write(f"    - {ex}\n")

print(f"Generated {len(unique_examples)} examples in {file_path}")
print("Intent: provide_medication_colour")
print("Entity: medication_colour")

Generated 400 examples in /home/anuuu/Documents/Work/Pillaxia/pillaxia-ai/data/medication_colour_nlu.yml
Intent: provide_medication_colour
Entity: medication_colour
